In [1]:
# boosting

In [2]:
# grandient boosting

In [3]:
import joblib 
df=joblib.load('../learning_notebooks/data/df.pkl')

In [4]:
df.head()

,Gender,ApplicantIncome,CoapplicantIncome,Loan_Amount,Loan_Amount_Term,Credit_History,TotalIncome,LoanIncomeRatio,Married_Yes,Dependents_1,Dependents_2,Dependents_3+,Education_Not Graduate,Self_Employed_Yes,Loan_Status_Y,IncomeGroup_medium,IncomeGroup_high,IncomeGroup_veryhigh
0,1,5849,0,146,360,1,5849,0,0,0,0,0,0,0,1,0,0,1
1,1,4583,1508,128,360,1,6091,0,1,1,0,0,0,0,0,0,0,1
2,1,3000,0,66,360,1,3000,0,1,0,0,0,0,1,1,1,0,0
3,1,2583,2358,120,360,1,4941,0,1,0,0,0,1,0,1,0,1,0
4,1,6000,0,141,360,1,6000,0,0,0,0,0,0,0,1,0,0,1


In [6]:
# prepare data
X=df.drop('Loan_Status_Y',axis=1)
y=df['Loan_Status_Y']

In [7]:
# train test split 
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [41]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier

num_feature=['Loan_Amount','TotalIncome']

preprocess=ColumnTransformer([
    ('num',StandardScaler(),num_feature)
])

gb_model=Pipeline([
    ('preprocess',preprocess), 
    ('classifier',GradientBoostingClassifier(
        n_estimators=100, 
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ))
])
gb_model.fit(X_train,y_train)

,steps,"[('preprocess', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [42]:
# prediction
y_pred_gb_model=gb_model.predict(X_test)
y_prob_gb_model=gb_model.predict_proba(X_test)[:,1]

In [43]:
# evalution
from sklearn.metrics import classification_report,confusion_matrix
print("\nconfusion_matrix : \n",confusion_matrix(y_test,y_pred_gb_model))
print("\nclassification_report : \n",classification_report(y_test,y_pred_gb_model))


confusion_matrix : 
 [[ 8 35]
 [14 66]]

classification_report : 
               precision    recall  f1-score   support

           0       0.36      0.19      0.25        43
           1       0.65      0.82      0.73        80

    accuracy                           0.60       123
   macro avg       0.51      0.51      0.49       123
weighted avg       0.55      0.60      0.56       123



In [44]:
# # feature importance
# import pandas as pd 
# pd.Series(
#     gb_model.feature_importances_, # not feature in gb_model
#     index=X.columns
# ).sort_values(ascending=False)

In [45]:
# observation
# learning rate - 0.1 - accuracy - 0.60
# learning rate - 0.5 - accuracy - 0.54
# learning rate - 0.9 - accuracy - 0.59
# learning rate - 1 - accuracy - 0.54

In [46]:
# xgboost model

In [47]:
# xtreme gradient boost model

In [48]:
# bulid model

In [50]:
from xgboost import XGBRFClassifier
xgb_model=XGBRFClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train,y_train)

,learning_rate,0.05
,subsample,0.8
,colsample_bynode,0.8
,reg_lambda,1
,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bytree,0.8
,device,None


In [51]:
# prediction
y_xgb_model_pred=xgb_model.predict(X_test)
y_xgb_model_prob=xgb_model.predict_proba(X_test)[:,1]

In [52]:
# evalution
print("\nconfusion_matrix : \n",confusion_matrix(y_test,y_pred_gb_model))
print("\nclassification_report : \n",classification_report(y_test,y_pred_gb_model))


confusion_matrix : 
 [[ 8 35]
 [14 66]]

classification_report : 
               precision    recall  f1-score   support

           0       0.36      0.19      0.25        43
           1       0.65      0.82      0.73        80

    accuracy                           0.60       123
   macro avg       0.51      0.51      0.49       123
weighted avg       0.55      0.60      0.56       123



In [53]:
# feature depend on split gain
xgb_model.feature_importances_

array([0.00651965, 0.02399189, 0.02955767, 0.02799973, 0.04183262,
       0.6135621 , 0.04430897, 0.        , 0.03945593, 0.02700161,
       0.01485311, 0.01342751, 0.0239331 , 0.01565522, 0.02891455,
       0.02145919, 0.02752718], dtype=float32)

In [54]:
# feature importance
import pandas as pd 
pd.Series(
    xgb_model.feature_importances_, 
    index=X.columns
).sort_values(ascending=False)

Credit_History            0.613562
TotalIncome               0.044309
Loan_Amount_Term          0.041833
Married_Yes               0.039456
CoapplicantIncome         0.029558
IncomeGroup_medium        0.028915
Loan_Amount               0.028000
IncomeGroup_veryhigh      0.027527
Dependents_1              0.027002
ApplicantIncome           0.023992
Education_Not Graduate    0.023933
IncomeGroup_high          0.021459
Self_Employed_Yes         0.015655
Dependents_2              0.014853
Dependents_3+             0.013428
Gender                    0.006520
LoanIncomeRatio           0.000000
dtype: float32

In [55]:
from sklearn.metrics import precision_score,recall_score,accuracy_score,f1_score
print("\nprecision_score : \n",precision_score(y_test,y_pred_gb_model))
print("\nrecall_score : \n",recall_score(y_test,y_pred_gb_model))
print("\nf1_score : \n",f1_score(y_test,y_pred_gb_model))
print("\naccuracy_score : \n",accuracy_score(y_test,y_pred_gb_model))


precision_score : 
 0.6534653465346535

recall_score : 
 0.825

f1_score : 
 0.7292817679558011

accuracy_score : 
 0.6016260162601627
